# EXP_021: Type-Conditioned ADASYN + Time-Decay 가중치 실험

이 노트북은 검사 유형별(Type-Conditioned) 독립 모델에 ADASYN 오버샘플링을 적용하고, 동시에 최근 데이터에 더 높은 Sample Weight를 부여하는 시계열 감쇠(Time-Decay) 가중치를 주입하여 성능을 테스트합니다.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import precision_recall_curve, auc, confusion_matrix
from imblearn.over_sampling import ADASYN
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

EXP_ID = "0824_dongjin_021_adasyn_time_decay"
DATA_PATH = "../data/raw/dataset.csv"
MODEL_PATH = f"../models/{EXP_ID}.pkl"

## 1. 데이터 로드 및 전처리
베이스라인과 동일하게 중복행을 제거하고 시간순(Timestamp)으로 분할합니다.

In [ ]:
# 데이터 로드
df = pd.read_csv(DATA_PATH)
if 'Unnamed: 0' in df.columns:
    df = df.rename(columns={'Unnamed: 0': 'record_id'})

# 중복 제거 (timestamp, record_id 제외한 컬럼 기준)
cols = [c for c in df.columns if c not in ['record_id', 'timestamp']]
df = df.drop_duplicates(subset=cols, keep='first').copy()

# 시계열 정렬
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

# 시계열 분할 (기존 baseline 분할 기준행 적용)
train_end = 235222
val_end = 313596

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

## 2. 모델 학습 (Type-Conditioned + ADASYN + Time-Decay)
각 inspection_type(0~4)별로 독립된 모델을 학습합니다.
- 데이터 내 Zero-variance 특성 제거
- Time-Decay 가중치 생성 (최근 14일마다 절반으로 감쇠)
- ADASYN 증식 및 가중치 병합

In [ ]:
models = {}
val_preds_list, val_y_list = [], []
test_preds_list, test_y_list = [], []

for t in [0, 1, 2, 3, 4]:
    print(f"\n--- Training Type {t} ---")
    tr = train_df[train_df['inspection_type'] == t].copy()
    va = val_df[val_df['inspection_type'] == t].copy()
    te = test_df[test_df['inspection_type'] == t].copy()
    
    if len(tr) == 0: continue
        
    y_tr, y_va, y_te = tr['class'], va['class'], te['class']
    
    # 의미 없는 컬럼 제거
    drop_cols = ['timestamp', 'inspection_type', 'class', 'record_id']
    drop_cols += [c for c in tr.columns if c.startswith('meta_feat') or c == 'Unnamed: 0']
    
    X_tr = tr.drop(columns=drop_cols, errors='ignore')
    X_va = va.drop(columns=drop_cols, errors='ignore')
    X_te = te.drop(columns=drop_cols, errors='ignore')
    
    # Zero variance 제거
    valid_cols = [c for c in X_tr.columns if X_tr[c].nunique() > 1]
    X_tr, X_va, X_te = X_tr[valid_cols], X_va[valid_cols], X_te[valid_cols]
    
    # Time-decay 가중치 계산
    max_ts = tr['timestamp'].max()
    delta_days = (max_ts - tr['timestamp']).dt.total_seconds() / (3600 * 24)
    decay_rate = 0.05
    sample_weights = np.exp(-decay_rate * delta_days).values
    
    # ADASYN 적용
    print(f"Original shape: {X_tr.shape}, Defects: {y_tr.sum()}")
    try:
        ada = ADASYN(random_state=42)
        X_ada, y_ada = ada.fit_resample(X_tr, y_tr)
        print(f"After ADASYN: {X_ada.shape}, Defects: {y_ada.sum()}")
    except Exception as e:
        print(f"ADASYN skipped for type {t} due to extreme sample lack.")
        X_ada, y_ada = X_tr, y_tr
        
    # 합성 데이터 가중치 부여 (1.0)
    weights = np.ones(len(y_ada))
    weights[:len(y_tr)] = sample_weights
    
    # XGBoost 학습
    clf = xgb.XGBClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=5,
        random_state=42, tree_method='hist'
    )
    clf.fit(X_ada, y_ada, sample_weight=weights, eval_set=[(X_va, y_va)], verbose=False)
    
    models[t] = {'model': clf, 'features': valid_cols}
    
    if len(X_va) > 0:
        val_preds_list.extend(clf.predict_proba(X_va)[:, 1])
        val_y_list.extend(y_va)
        
    if len(X_te) > 0:
        test_preds_list.extend(clf.predict_proba(X_te)[:, 1])
        test_y_list.extend(y_te)

os.makedirs('../models', exist_ok=True)
joblib.dump(models, MODEL_PATH)

## 3. 평가 및 지표 추출
전체 Validation 결과로부터 최적의 Threshold를 찾고, 최종 Test 성능을 도출합니다.

In [ ]:
val_y = np.array(val_y_list)
val_probs = np.array(val_preds_list)

test_y = np.array(test_y_list)
test_probs = np.array(test_preds_list)

# 최적 Threshold 탐색
precisions, recalls, thresholds = precision_recall_curve(val_y, val_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
best_th = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
if best_th > 0.99: best_th = 0.5

# Test Set 평가
test_prec, test_rec, _ = precision_recall_curve(test_y, test_probs)
test_pr_auc = auc(test_rec, test_prec)

test_preds = (test_probs >= best_th).astype(int)
tn, fp, fn, tp = confusion_matrix(test_y, test_preds).ravel()

recall = tp / (tp + fn) if (tp + fn) > 0 else 0
fcr = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"Test PR-AUC: {test_pr_auc:.4f}")
print(f"Best Threshold: {best_th:.4f}")
print(f"Recall: {recall:.4f}")
print(f"False Call Reduction: {fcr:.4f}")
print(f"Confusion Matrix: TN={tn} FP={fp} FN={fn} TP={tp}")